# 텍스트 데이터 다루기

## 텍스트 토큰화하기
LLM을 위한 임베딩을 만드는 데 필수적인 전처리 단계인, 입력 텍스트를 개별 토큰으로 분할하는 방법에 대해 알아보자.

LLM 훈련을 위해 토큰화할 텍스트는 이디스 워튼의 단편 소설인 [The Verdict(심판)]이다.



In [2]:
import urllib.request
url = ("https://raw.githubusercontent.com/rickiepark/"
       "llm-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()
print("총 문자 개수:", len(raw_text))
print(raw_text[:99])

총 문자 개수: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


목표는 20,479 문자로 이루어진 이 단편 소설을 개별 단어와 특수 문자로 토큰화하는 것이다.

이디스 워튼의 소설 전체에 기본적인 토크나이저를 적용해보자.

In [3]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(len(preprocessed)) # (공백을 제외한) 텍스트에 있는 토큰의 개수
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 토큰을 토큰 ID로 변환하기
다음으로 토큰을 파이썬 문자열에서 정수 표현으로 바꾸어 토큰 ID를 만들어 보자.

이 변환은 토큰 ID를 임베딩 벡터로 변환하기 전의 중간 단계이다.

앞서 생성한 토큰을 토큰ID로 매핑하려면 어휘사전(vocabulary)를 먼저 구축해야 한다.

토큰화된 이디스 워튼의 단편 소설이 파이썬 변수 preprocessed에 저장되어 있으므로 모든 고유 토큰의 리스트를 만들고 알파벳 순으로 정렬하여 어휘 사전의 크기를 확인해본다.

In [4]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


어휘사전을 만든 후 처음 51개 항목을 출력해보자.

In [5]:
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
  print(item)

  if(i>=50):
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


여기서 볼 수 있듯이 이 딕셔너리(dictionary)는 개별 토큰과 이에 연관된 고유한 정수 레이블을 담고 있다.

다음 목표는 이 어휘사전을 새로운 텍스트에 적용하여 토큰 ID로 변환하는 것이다.

LLM의 출력을 숫자에서 텍스트로 변환할 때 토큰 ID를 텍스트로 바꿀 방법이 필요하다. 이를 위해 어휘사전을 뒤집어 토큰 ID를 텍스트 토큰으로 매핑해야 한다.

파이썬으로 완전한 토크나이저를 구현해보자. 이 클래스는 텍스트를 토큰으로 분할하고 어휘사전으로 문자열-정수 매핑을 수행해 토큰 ID를 생성하는 encode 메서드를 가진다. 또한 토큰 ID를 텍스트로 변환하기 위해 역방향으로 정수-문자열 매핑을 수행하는 decode 메서드도 구현한다.

다음은 이 토크나이저를 구현하는 코드이다.

In [6]:
class SimpleTokenizerV1:
  def __init__(self, vocab):
    # encode 메서드와 decode 메서드에서 참조할 수 있도록 어휘사전을 클래스의 속성으로 저장
    self.str_to_int = vocab
    # 토큰 ID를 원본 텍스트 토큰으로 매핑하는 역어휘사전을 만듦
    self.int_to_str = {i:s for s,i in vocab.items()}

  def encode(self, text): # 입력 텍스트를 처리하여 토큰 ID로 변환
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids): # 토큰 ID를 텍스트로 되돌림
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'([,.:;?_!"()\']|--|\s)', r'\1', text) # 지정된 구두점 문자 앞의 공백을 삭제
    return text

SimpleTokenizerV1 파이썬 클래스에 기존의 어휘사전을 전달하여 새로운 토크나이저 객체를 생성한다.

SimpleTokenizerV1 클래스로 새로운 토크나이저 객체를 만들고 이디스 워튼의 단편 소설의 한 구절을 토큰화해 보자.

In [7]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know, "
       Mrs. Gisburn said with pardonable pride. """
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


decode 메서드로 이 토큰 ID를 다시 텍스트로 바꿀 수 있는지 확인해보자.

In [8]:
print(tokenizer.decode(ids))

" It ' s the last he painted , you know , " Mrs . Gisburn said with pardonable pride .


## 특수 문맥 토큰 추가하기
알지 못하는 단어를 처리하기 위해서는 토크나이저를 수정해야 한다. 모델이 텍스트로부터 문맥이나 그 밖의 정보를 잘 이해할 수 있도록 특수 문맥 토큰도 추가해야 한다.

이런 특수 토큰은 알지 못하는 단어, 문서 경계 등을 표시하는 데 사용된다.

토크나이저가 어휘사전에 없는 단어를 만났을 때 <|unk|> 토큰을 사용하도록 수정한다. 또한 관련이 없는 텍스트 사이에 <|endoftext|> 토큰을 추가한다.


이렇게 하면 훈련을 위해 텍스트가 연결되어 있지만 사실 관련이 없다는 것을 LLM이 이해하는 데 도움이 되기 때문이다.

그럼 이 두 특수 토큰 <|unk|>와 <|endoftext|>를 고유 단어 목록에 추가하여 어휘사전을 수정해보자.

In [9]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>" ,"<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [10]:
class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = { i:s for s, i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    # 알지 못하는 단어를 <|unk|> 토큰을 바꾼다.
    preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    # 구두점 문자 앞의 공백을 삭제한다.
    text = re.sub(r'([,.:;?_!"()\']|--|\s)', r'\1', text)
    return text


SimpleTokenizerV1과 비교해 보면 새로운 SimpleTokenizerV2는 알지 못하는 단어를 <|unk|> 토큰으로 바꾼다.

서로 관련이 없는 2개의 독립된 문장을 연결한 간단한 텍스트 샘플에 이 새로운 토크나이저를 사용해보자.

In [11]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = "<|endoftext|>".join((text1, text2))
print(text)

Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.


In [12]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

[1131, 5, 355, 1126, 628, 975, 10, 1131, 988, 956, 984, 722, 988, 1131, 7]
<|unk|> , do you like tea ? <|unk|> the sunlit terraces of the <|unk|> .


## 바이트 페어 인코딩
바이트 페어 인코딩(BPE) 기반의 고급 토큰화 방법을 알아보자.

BPE 구현은 상대적으로 복잡하기 때문에 파이썬 오픈 소스 라이브러리인 tiktoken을 사용한다.

In [13]:
from importlib.metadata import version
import tiktoken
print("tiktoken 버전:", version("tiktoken"))

tiktoken 버전: 0.13.0


In [14]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [15]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


토큰 ID와 디코딩된 텍스트를 바탕으로 두 가지 중요한 점을 관찰할 수 있다.

첫째, <|endoftext|> 토큰은 50256과 같이 비교적 큰 토큰 ID에 할당된다. 사실 ChatGPT의 초기 모델을 훈련하는 데 사용되는 BPE 토크나이저는 50,257 크기의 어휘사전을 가지고 있다. 그래서 <|endoftext|>에 가장 큰 토큰 ID가 할당된다.

둘째, BPE 토크나이저는 someunknownPlace와 같은 알지 못하는 단어를 정확하게 인코딩하고 디코딩한다. BPE 토크나이저는 <|unk|> 토큰을 사용하지 않고 어떻게 알지 못하는 어떤 단어도 처리할 수 있는 걸까?

BPE 알고리즘은 어휘사전에 없는 단어를 더 작은 부분단어, 심지어 개별 문자로 나누어 처음 본 단어를 처리한다. 알지 못하는 단어를 개별 문자로 분할하는 기능 덕분에 토크나이저와 이런 토크나이저로 훈련된 LLM이 훈련 데이터에 없는 단어가 포함되어 있더라도 모든 텍스트를 처리할 수 있다.

## 슬라이딩 윈도로 데이터 샘플링하기

LLM을 위한 임베딩을 만드는 다음 단계는 LLM 훈련에 필요한 입력-타깃 쌍을 생성하는 것이다.

슬라이딩 윈도를 사용해 훈련 데이터셋에서 입력-타깃 쌍을 추출하는 데이터로더를 구현해보자.

먼저 BPE 토크나이저로 소설 [The Verdict] 전체를 토큰화한다.

In [16]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


다음으로 조금 더 흥미로운 텍스트 구절을 만들기 위해 데이터셋에 있는 처음 50개 토큰을 삭제한다.

In [17]:
enc_sample = enc_text[50:]

다음 단어 예측 작업을 위해 입력-타깃 쌍을 만드는 가장 쉽고 직관적인 방법 중 하나는 입력 토큰을 담은 x와 입력에서 토큰 하나만큼 이동한 타깃을 담은 y 변수를 만드는 것이다.

In [18]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1: context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


입력과 토큰 하나만큼 이동시킨 타깃을 사용해 다음 단어 예측 작업을 구성할 수 있다.

In [19]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context, "--->", desired)

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


토큰 ID를 텍스트로 바꾸도록 앞의 코드를 다시 작성해 보자.

In [20]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "--->", tokenizer.decode([desired]))

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


이제 LLM 훈련을 위한 입력-타깃 쌍을 만들었다.

토큰을 임베딩으로 바꾸기 전에 한 가지 작업이 더 남았다. 입력 데이터셋을 순회하면서 파이토치 텐서로 입력과 타깃을 반환하는 효율적인 데이터로더를 구현해야 한다.

데이터셋 클래스를 위한 코드는 다음과 같다.

In [21]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt) # 전체 텍스트를 토큰화

    # 슬라이딩 윈도우를 사용해 책을 max_length 길이의 중첩된 시퀀스로 나눔
    for i in range(0, len(token_ids)-max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  # 데이터셋에 있는 전체 행 수를 반환
  def __len__(self):
    return len(self.input_ids)

  # 데이터셋에서 하나의 행을 반환
  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

GPTDatasetV1 클래스는 파이토치 Dataset 클래스를 기반으로 하며 데이터셋에서 개별 행을 추출하는 방법을 정의한다. 각 행은 input_chunk 텐서에 할당된 (max_length만큼) 여러 개의 토큰 ID로 구성된다. target_chunk 텐서는 각 행에 상응하는 타깃을 가지고 있다.

다음 코드는 GPTDatasetV1을 사용하여 파이토치 DataLoader를 통해 입력을 배치로 로드한다.

In [22]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
  tokenizer = tiktoken.get_encoding("gpt2") # 토크나이저 초기화
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # 데이터셋 만듦
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle = shuffle,
      # drop_last=True로 지정하면 batch_size보다 작을 경우
      # 훈련 손실이 갑자기 높아지는 것을 피하기 위해 마지막 배치 삭제
      drop_last=drop_last,
      # 전처리에 사용할 CPU 프로세서 개수
      num_workers=num_workers
  )

  return dataloader

GPTDatasetV1 클래스와 create_dataloader_v1 함수가 어떻게 동작하는지 이해하기 위해 문맥 크기를 4와 배치 크기 1로 dataloader를 테스트해보자.

In [25]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


first_batch 변수는 2개의 텐서를 담고 있다.

첫 번째 텐서는 입력 토큰 ID를 저장하고 있고, 두 번째 텐서는 타깃 토큰 ID를 저장하고 있다. max_length가 4이므로 두 텐서는 4개의 토큰 ID를 가지고 있다.

일반적으로 LLM을 훈련할 때는 적어도 256 크기의 입력을 사용한다.

stride=1의 의미를 이해하기 위해 이 데이터셋에서 또 다른 배치를 추출해보자.

In [26]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


첫 번째 배치와 두 번째 배치를 비교해 보면 두 번째 배치의 토큰 ID가 하나씩 밀려 있다는 것을 알 수 있다.

지금까지 배치 크기 1은 설명하기에 좋다. 작은 배치 크기는 훈련 과정에서 메모리를 덜 필요로 하지만 모델 업데이트에 잡음이 더 많다. 배치 크기에는 트레이드 오프가 있고 LLM을 훈련할 때 실험해봐야 할 하이퍼파라미터이다.

배치 크기가 1보다 클 경우 데이터 로더로 샘플링하는 방법을 간단히 살펴보자.

In [27]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("입력:\n", inputs)
print("\n타깃:\n", targets)

입력:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

타깃:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


데이터셋을 완전히 활용하기 위해 스트라이드를 4로 늘렸다. (한 토큰도 건너뛰지 않음)

배치 사이에 중첩이 있으면 과대적합이 증가할 수 있는데 이렇게 하면 과대적합을 피할 수 있다.

## 토큰 임베딩 만들기
LLM 훈련을 위한 입력 텍스트 준비의 마지막 단계는 토큰 ID를 임베딩 벡터로 변환하는 것이다.

준비 단계에서는 이런 임베딩 벡터를 랜덤한 값으로 초기화한다. 이런 초기화는 LLM 학습 과정의 시작점 역할을 한다.

GPT와 같은 LLM은 역전파(backpropagation) 알고리즘으로 훈련되는 심층 신경망이므로 연속적인 벡터 표현인 임베딩이 필수적이다.

예시를 통해 토큰 ID를 임베딩 벡터로 변환하는 방법을 알아보자. 다음과 같은 4개의 입력 토큰(토큰 ID가 2, 3, 5, 1)이 있다고 가정하자.

In [28]:
input_ids = torch.tensor([2, 3, 5, 1])

간단한 설명을 위해 (GPT-2의 BPE 토크나이저에 있는 50,257개의 단어로 구성된 어휘사전 대신) 단 6개의 단어로 구성된 작은 어휘사전을 가정해보자.

크기가 3인 임베딩을 만든다.

vocab_size와 output_dim을 사용해 파이토치의 임베딩 층을 초기화할 수 있다.

In [29]:
vocab_size = 6
output_dim=3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


임베딩 층의 가중치 행렬은 작고 랜덤한 값을 담고 있다. 이 값은 LLM 최적화의 일부로, LLM 훈련 과정에서 최적화된다.

또한 이 가중치 행렬은 행이 6개이고 열이 3개이다. 어휘 사전에 있는 6개의 토큰 각각에 하나의 행이 할당되고 3개의 임베딩 차원 각각에 하나의 열이 할당된다.

이를 토큰 ID에 적용하여 임베딩 벡터를 얻어보자.
반환된 임베딩 벡터는 다음과 같다.

In [30]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


토큰 ID 3의 임베딩 벡터를 앞의 임베딩 행렬과 비교해 보면 네 번째 행의 값과 같다는 것을 알 수 있다. (파이썬의 인덱스는 0부터 시작하므로 인덱스 3에 해당하는 행이 네 번째이다.)

다른 말로 하면 임베딩 층은 토큰 ID를 기반으로 가중치 행렬에서 행을 추출하는 검색 연산을 수행한다.

하나의 토큰 ID를 3차원 임베딩 벡터로 바꾸는 방법을 알아보았다. 이제 4개의 입력 ID (torch.tenxor([2, 3, 5, 1])에 이를 모두 적용해보자.

In [31]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


토큰 ID로 임베딩 벡터를 만들었으니 다음으로 임베딩 벡터를 조금 수정하여 텍스트 안에 있는 토큰의 위치 정보를 인코딩하자.

## 단어 위치 인코딩하기

원칙적으로 토큰 임베딩은 LLM의 입력으로 적합하다. 하지만 LLM의 단점 중 하나는 시퀀스 안의 토큰 위치 또는 순서에 대한 개념이 셀프 어텐션 메커니즘에 없다는 것이다.

앞서 소개한 임베딩 층의 작동 방식은 입력 시퀀스 안에 토큰 ID의 위치에 상관없이 동일한 토큰 ID를 항상 동일한 벡터 표현에 매핑한다.

원칙적으로, 결정론적이고 위치에 독립적인 토큰 ID의 임베딩은 재현 가능성의 목적으로 좋다. 하지만 LLM의 셀프 어텐션 메커니즘 자체가 위치에 구애받지 않기 때문에 LLM에 추가적인 위치 정보를 주입하는 것이 도움이 된다.

이를 위해 크게 두 종류의 위치를 고려한 임베딩을 사용할 수 있다. 상대위치 임베딩과 절대 위치 임베딩이다.

절대 위치 임베딩은 시퀀스에 특정 위치에 직접 연관된다. 입력 시퀀스의 각 위치에 대해서 고유한 임베딩이 토큰 임베딩에 더해져 정확한 위치 정보를 추가한다.

상대 위치 임베딩은 토큰의 절대 위치에 초점을 맞추는 대신 상대적인 위치 또는 토큰 사이의 거리를 강조한다. 이는 모델이 정확한 위치가 아니라 멀리 떨어진 정도를 바탕으로 관계를 학습한다는 의미이다. 이런 방식은 모델이 길이가 다른 시퀀스에도 더 잘 일반화될 수 있다는 것이다.

두 종류의 위치 임베딩은 LLM이 토큰 사이의 순서와 관계를 이해하는 능력을 보강하여 정확하고 맥락을 고려한 예측을 만드는 데 목적이 있다. 둘 중 어느 것을 선택하느냐는 애플리케이션과 처리하려는 데이터의 성질에 따라 달라지는 경우가 많다.

오픈AI의 GPT 모델은 원본 트랜스포머 모델의 위치 임베딩과 같이 고정되거나 사전에 정의된 임베딩이 아니라 훈련 과정에서 최적화되는 절대 위치 임베딩을 사용한다. 이 최적화 과정은 모델 훈련의 일부로 수행된다.

지금은 초깃값으로 채워진 위치 임베딩을 만들어 LLM 입력을 준비해보자.

앞서 간단하게 하려고 매우 작은 임베딩 크기를 선택했다. 이제 조금 더 현실적이고 유용한 임베딩 크기를 선택해서 입력 토큰을 256차원의 벡터 표현으로 인코딩해보자. 원본 GPT-3 모델이 사용한 차원(GPT-3의 임베딩 크기는 12,288차원)보다는 작지만 실험에는 적합한 크기이다. 또한 토큰 ID를 앞서 구현한 50,257 크기의 어휘사전을 가진 BPE 토크나이저로 만들었다고 가정하자.

In [32]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

token_embedding_layer를 사용해 데이터 로더를 통해 샘플링한 각 배치에 있는 토큰 ID를 256 차원 벡터로 임베딩한다. 배치 크기가 8이고 4개의 토큰씩 들어있다면 만들어진 벡터는 8 x 4 x 256 텐서가 될 것이다.

먼저 데이터 로더를 초기화한다.

In [33]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("토큰 ID:\n", inputs)
print("\n입력 크기:\n", inputs.shape)

토큰 ID:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

입력 크기:
 torch.Size([8, 4])


여기서 보듯이 토큰 ID 텐서의 차원이 8 x 4이다. 배치에 4개의 토큰을 가진 텍스트 샘플 8개가 들어있다는 의미이다.

임베딩 층을 사용해 이 토큰 ID를 256 차원의 벡터로 임베딩해보자.

In [34]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


8 x 4 x 256 차원의 텐서는 각 토큰 ID가 256 차원의 벡터로 임베딩되었다는 것을 보여준다.

GPT 모델의 절대 임베딩 방법에서는 token_embedding_layer와 동일한 임베딩 차원을 가지는 또 다른 임베딩 층을 만들면 된다.

In [35]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


여기서 보듯이 위치 임베딩 벡터는 4개의 256차원 벡터로 구성된다. 이제 이 벡터를 토큰 임베딩에 바로 더할 수 있다.

파이토치는 4 x 256차원의 pos_embeddings 텐서를 배치에 있는 4 x 256 차원의 토큰 임베딩 텐서 8개에 각각 더한다.


In [36]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
